In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
df=pd.read_csv('../data/loans_cleaned.csv')
df.head()

,log.annual.inc,dti,fico,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_major_purchase,purpose_small_business,revol.bal_to_income,years_with_cr_line
0,11.350407,19.48,737,28854.0,52.1,0.0,0.0,0.0,0,0,1,0,0,0,0,0.339459,15.451941
1,11.082143,14.29,707,33623.0,76.7,0.0,0.0,0.0,0,1,0,0,0,0,0,0.517277,7.561644
2,10.373491,11.63,682,3511.0,25.6,1.0,0.0,0.0,0,0,1,0,0,0,0,0.109719,12.904110
3,11.350407,8.10,712,33667.0,73.2,1.0,0.0,0.0,0,0,1,0,0,0,0,0.396082,7.397146
4,11.299732,14.97,667,4740.0,39.5,0.0,1.0,0.0,0,1,0,0,0,0,0,0.058663,11.139726


In [5]:
X= df.drop(columns=['not.fully.paid'])
y=df['not.fully.paid']

In [6]:
X_train,X_test, y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train default rate:", y_train.mean().round(4))
print("Test default rate:", y_test.mean().round(4))



Train shape: (7662, 16)
Test shape: (1916, 16)
Train default rate: 0.16
Test default rate: 0.1602


In [9]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(scale_pos_weight)

5.249592169657422


In [10]:
import xgboost as xgb

model= xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate= 0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

In [11]:
model.fit(X_train,y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [16]:
y_pred= model.predict(X_test)

In [14]:
y_pred_proba= model.predict_proba(X_test)[:,1]
y_pred_proba

array([0.53653425, 0.41608086, 0.24782304, ..., 0.47088823, 0.296401  ,
       0.4019376 ], shape=(1916,), dtype=float32)

In [19]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
auc= roc_auc_score(y_test, y_pred_proba)
auc

0.6497612169332521

In [20]:
classification_report(y_test,y_pred)

'              precision    recall  f1-score   support\n\n           0       0.88      0.71      0.78      1609\n           1       0.24      0.47      0.32       307\n\n    accuracy                           0.67      1916\n   macro avg       0.56      0.59      0.55      1916\nweighted avg       0.77      0.67      0.71      1916\n'

In [21]:
confusion_matrix(y_test,y_pred)

array([[1142,  467],
       [ 162,  145]])

In [23]:
threshold = 0.3
y_pred_lower = (y_pred_proba >= threshold).astype(int)
y_pred_lower

array([1, 1, 0, ..., 1, 0, 1], shape=(1916,))

In [24]:
print(classification_report(y_test, y_pred_lower))

              precision    recall  f1-score   support

           0       0.94      0.23      0.37      1609
           1       0.19      0.93      0.31       307

    accuracy                           0.34      1916
   macro avg       0.56      0.58      0.34      1916
weighted avg       0.82      0.34      0.36      1916



In [25]:
print(confusion_matrix(y_test, y_pred_lower))


[[ 371 1238]
 [  23  284]]


In [26]:
threshold = 0.4
y_pred_04 = (y_pred_proba >= threshold).astype(int)
print(classification_report(y_test, y_pred_04))
print(confusion_matrix(y_test, y_pred_04))

              precision    recall  f1-score   support

           0       0.90      0.43      0.59      1609
           1       0.20      0.76      0.32       307

    accuracy                           0.49      1916
   macro avg       0.55      0.60      0.45      1916
weighted avg       0.79      0.49      0.54      1916

[[697 912]
 [ 74 233]]


In [27]:
param_grid = {
    'max_depth': [3, 4, 6],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
}

In [28]:
from sklearn.model_selection import GridSearchCV
base_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\nBest params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

best_model = grid_search.best_estimator_
best_pred_proba = best_model.predict_proba(X_test)[:, 1]
best_auc = roc_auc_score(y_test, best_pred_proba)
print("Test AUC with best model:", best_auc)

Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best CV AUC: 0.6600021002667787
Test AUC with best model: 0.6612195650281498


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train_imputed, y_train)
rf_pred_proba = rf_model.predict_proba(X_test_imputed)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred_proba)
print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.6598125770553664


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

log_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_pred_proba = log_model.predict_proba(X_test_scaled)[:, 1]
log_auc = roc_auc_score(y_test, log_pred_proba)
print("Logistic Regression AUC:", log_auc)

print("\n--- Summary ---")
print("Random Forest:", rf_auc)
print("Logistic Regression:", log_auc)

Logistic Regression AUC: 0.6752368092346998

--- Summary ---
Random Forest: 0.6598125770553664
Logistic Regression: 0.6752368092346998


In [38]:
from sklearn.pipeline import Pipeline
final_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

final_pipeline.fit(X_train, y_train)

final_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]
final_auc = roc_auc_score(y_test, final_pred_proba)
print("\nFinal pipeline AUC (sanity check, should match Logistic Regression above):", final_auc)



Final pipeline AUC (sanity check, should match Logistic Regression above): 0.6752368092346998


In [40]:
import joblib

joblib.dump(final_pipeline, "../models/loan_default_pipeline.joblib")
joblib.dump(list(X.columns), "../models/feature_names.joblib")
print("Saved.")

Saved.
